In [ ]:
cpip install xgboost pandas numpy scikit-learn tensorflow

In [ ]:
#Use tensorflow to build gradient booster for mid game updates, predicting which team is going to win the game.
#NHL api for live update data. retrieveplaybyplay data file has code for taking it.
#Goals scored. shots on goal. possession of puck. which side of the arena on other side. player rating. money stuff.
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler


In [ ]:
#Troy testing cell

import requests
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


def time_to_float(time_str):
    minutes, seconds = map(int, time_str.split(":"))
    return minutes + seconds / 60


column_index = {
    "timeRemaining": 0, "homeScore": 1, "awayScore": 2, "homeSog": 3, "awaySog": 4,
    "homePIM": 5, "awayPIM": 6, "goalDiff": 7, "xFaceoff": 8, "goalDiff_timeRemaining": 9
}


def fetch_nhl_live_data(season):
    game_count = 1

    all_games = []
    all_labels = []

    while True:

        game_id = f"{season}02{game_count:04}"

        if game_count % 100 == 0:
            print(f"Processed {game_count} games...")

        url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
        res = requests.get(url)

        if res.status_code == 404:

            print("Error 404: Not Found")
            break

        data = res.json()
        game_df, y = working_data(data)

        if game_df is not None:
            all_games.append(game_df)
            all_labels.append(y)

        game_count += 1

    all_games_df = pd.concat(all_games, ignore_index=True)
    all_labels_arr = np.concatenate(all_labels)

    all_games_df["winner"] = all_labels_arr
    all_games_df.to_csv("all_games.csv", index=False)
   # all_labels_df.to_csv("all_labels.csv", index=False)


    return all_games_df, all_labels_arr


def working_data(game_data):
    if not game_data:
        return None, None

    home_fo = 0
    away_fo = 0

    game = []

    home_id = game_data["homeTeam"]["id"]
    count = 0

    current_state = np.zeros(len(column_index))

    for event in game_data["plays"]:

        if count % 4 == 0:
            state = current_state.copy()
            time_in_min = time_to_float(event["timeInPeriod"])
            period = event["periodDescriptor"]["number"]
            state[column_index["timeRemaining"]] = max(0, 60 - (time_in_min * period))

            match event["typeCode"]:
                case 502:  # Faceoff
                    teamId = event["details"].get("eventOwnerTeamId")
                    if teamId == home_id:
                        home_fo += 1
                    else:
                        away_fo += 1
                    if (home_fo + away_fo) > 0:
                        state[column_index["xFaceoff"]] = home_fo / (home_fo + away_fo)

                case 505:  # Goal
                    teamId = event["details"].get("eventOwnerTeamId")
                    if teamId == home_id:
                        state[column_index["homeScore"]] += 1
                    else:
                        state[column_index["awayScore"]] += 1
                    state[column_index["goalDiff"]] = state[column_index["homeScore"]] - state[column_index["awayScore"]]

                case 506:  # Shot on Goal
                    teamId = event["details"].get("eventOwnerTeamId")
                    if teamId == home_id:
                        state[column_index["homeSog"]] += 1
                    else:
                        state[column_index["awaySog"]] += 1

                case 509:  # Penalty
                    teamId = event["details"].get("eventOwnerTeamId")
                    if teamId == home_id:
                        state[column_index["homePIM"]] += event["details"]["duration"]
                    else:
                        state[column_index["awayPIM"]] += event["details"]["duration"]

            if state[column_index["timeRemaining"]] > 0:
                state[column_index["goalDiff_timeRemaining"]] = state[column_index["goalDiff"]] / state[column_index["timeRemaining"]]
            else:
                state[column_index["goalDiff_timeRemaining"]] = state[column_index["goalDiff"]]

            game.append(state)
            current_state = state.copy()

        count += 1

    if not game:
        return None, None

    df = pd.DataFrame(game, columns=column_index.keys())

    label = 1 if game_data["homeTeam"]["score"] >= game_data["awayTeam"]["score"] else 0
    labels = np.full((len(df),), label)

    return df, labels


# Training

season = 2023
df, labels = fetch_nhl_live_data(season)
df = pd.read_csv('/content/all_games.csv')

X = df.drop(columns=['winner'])
y = df['winner']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.2, max_depth=9)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Model Accuracy:", accuracy_score(y_test, y_pred))




Processed 100 games...
Processed 200 games...
Processed 300 games...
Processed 400 games...
Processed 500 games...
Processed 600 games...
Processed 700 games...
Processed 800 games...
Processed 900 games...
Processed 1000 games...
Processed 1100 games...
Processed 1200 games...
Processed 1300 games...
Error 404: Not Found
Model Accuracy: 0.7524299874891733


In [ ]:
#Troy single game thing

# Troy single game thing

def predict_game_win_probabilities(game_id, model):

    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
    res = requests.get(url)

    if res.status_code != 200:
        print(f"Game ID {game_id} not found or not available.")
        return

    game_data = res.json()
    game_df, _ = working_data(game_data)

    if game_df is None or game_df.empty:
        print("No data could be processed for this game.")
        return

    probs = model.predict_proba(game_df)

    print(f"\nWin probabilities throughout game {game_id}:\n")
    print(f"{'Minute':>8} | {'Home Win %':>10} | {'Away Win %':>10}")
    print("-" * 35)

    for i, row in game_df.iterrows():
        minute = 60 - row['timeRemaining']

        home_win_prob = probs[i][1] * 100
        away_win_prob = probs[i][0] * 100
        print(f"{minute:8.2f} | {home_win_prob:10.2f} | {away_win_prob:10.2f}")




In [ ]:
predict_game_win_probabilities("2023020034", model)



Win probabilities throughout game 2023020034:

  Minute | Home Win % | Away Win %
-----------------------------------
    0.00 |      54.51 |      45.49
    0.80 |      54.91 |      45.09
    1.42 |      52.41 |      47.59
    2.20 |      52.93 |      47.07
    3.35 |      49.83 |      50.17
    3.60 |      49.83 |      50.17
    4.03 |      49.23 |      50.77
    5.00 |      50.24 |      49.76
    5.22 |      49.53 |      50.47
    5.53 |      51.51 |      48.49
    5.80 |      50.99 |      49.01
    6.87 |      49.32 |      50.68
    8.20 |      49.19 |      50.81
    9.05 |      50.44 |      49.56
    9.18 |      51.47 |      48.53
    9.77 |      54.65 |      45.35
   10.05 |      55.80 |      44.20
   10.95 |      48.75 |      51.25
   11.77 |      53.88 |      46.12
   12.10 |      53.88 |      46.12
   13.50 |      53.88 |      46.12
   14.05 |      53.88 |      46.12
   14.83 |      53.34 |      46.66
   16.78 |      51.27 |      48.73
   17.25 |      56.23 |      43.77
   18.

In [ ]:



pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

team_name_dict={
    'New York Rangers' : 'NYR',
    'Winnipeg Jets' : 'WPG',
    'Montreal Canadiens' : 'MTL',
    'Los Angeles Kings' : 'LAK',
    'Anaheim Ducks' : 'ANA',
    'Buffalo Sabres' : 'BUF',
    'Carolina Hurricanes' : 'CAR',
    'Utah Hockey Club' : 'UTA',
    'Philadelphia Flyers' : 'PHI',
    'New York Islanders' : 'NYI',
    'Tampa Bay Lightning' : 'TBL',
    'Pittsburgh Penguins' : 'PIT',
    'Ottawa Senators' : 'OTT',
    'Calgary Flames' : 'CGY',
    'Boston Bruins' : 'BOS',
    'Colorado Avalanche' : 'COL',
    'Columbus Blue Jackets' : 'CBJ',
    'Florida Panthers' : 'FLA',
    'Vancouver Canucks' : 'VAN',
    'Chicago Blackhawks' : 'CHI',
    'San Jose Sharks' : 'SJS',
    'Washington Capitals' : 'WSH',
    'St. Louis Blues' : 'STL',
    'Dallas Stars' : 'DAL',
    'Toronto Maple Leafs' : 'TOR',
    'Edmonton Oilers' : 'EDM',
    'New Jersey Devils' : 'NJD',
    'Seattle Kraken' : 'SEA',
    'Vegas Golden Knights' : 'VGK',
    'Detroit Red Wings' : 'DET',
    'Nashville Predators' : 'NSH',
    'Minnesota Wild' : 'MIN',
    'Arizona Coyotes' : 'ARI'
}

dropped_items = [
    'Date', 'Start Time (Sask)',
    'Start Time (ET)', 'Status',
    'home_team', 'home_win',
    'Home_Score', 'home_situation',
    'home_season', 'home_name',
    'home_team.1', 'home_position',
    'home_xFreezeFor', 'home_xPlayStoppedFor',
    'home_freezeFor', 'home_freezeAgainst',
    'home_playStoppedAgainst',
    'away_team',
    'Away_Score', 'away_situation',
    'away_season', 'away_name',
    'away_team.1', 'away_position',
    'away_xFreezeFor', 'away_xPlayStoppedFor',
    'away_freezeFor', 'away_freezeAgainst',
    'away_playStoppedAgainst', 'Visitor', 'Home',
    'home_iceTime',

    'home_xGoalsFor', 'home_xGoalsAgainst',
    'home_corsiPercentage', 'home_scoreAdjustedTotalShotCreditFor',
    'home_scoreAdjustedTotalShotCreditAgainst',
    'home_scoreFlurryAdjustedTotalShotCreditAgainst',
    'home_scoreFlurryAdjustedTotalShotCreditFor',
    'home_scoreFlurryAdjustedTotalShotCreditAgainst',
    'away_xGoalsFor', 'away_xGoalsAgainst',
    'away_corsiPercentage', 'away_scoreAdjustedTotalShotCreditFor',
    'away_scoreAdjustedTotalShotCreditAgainst',
    'away_scoreFlurryAdjustedTotalShotCreditAgainst',
    'away_scoreFlurryAdjustedTotalShotCreditFor',
    'away_scoreFlurryAdjustedTotalShotCreditAgainst'

]
# Load dataset
team_data_24_25 = pd.read_csv('/content/2024_25_NHL_Team_Stats.csv')
team_data_23_24 = pd.read_csv('/content/teams.csv')
game_scores_24_25 = pd.read_csv('/content/nhl-202425-asplayed.csv')
game_scores_23_24 = pd.read_csv('/content/nhl-202324-asplayed.csv')

game_scores_24_25.columns = ['Date', 'Start Time (Sask)', 'Start Time (ET)', 'Visitor', 'Away_Score', 'Home', 'Home_Score', 'Status']
game_scores_23_24.columns = ['Date', 'Start Time (Sask)', 'Start Time (ET)', 'Visitor', 'Away_Score', 'Home', 'Home_Score', 'Status']

# Filter to include only rows up to the last row with a valid Away_Score
game_scores_24_25 = game_scores_24_25[game_scores_24_25['Away_Score'].notna()]
game_scores_23_24 = game_scores_23_24[game_scores_23_24['Away_Score'].notna()]

team_data_24_25 = team_data_24_25[team_data_24_25['situation'] == '5on5']
team_data_23_24 = team_data_23_24[team_data_23_24['situation'] == '5on5']

# Label 1 for home win 0 for away win
game_scores_24_25['home_win'] = (game_scores_24_25['Home_Score'] > game_scores_24_25['Away_Score']).astype(int)

game_scores_24_25['Visitor'] = game_scores_24_25['Visitor'].map(team_name_dict)

game_scores_24_25['Home'] = game_scores_24_25['Home'].map(team_name_dict)


team_data_23_24 = team_data_23_24[team_data_23_24['situation'] == '5on5']

# Label 1 for home win 0 for away win
game_scores_23_24['home_win'] = (game_scores_23_24['Home_Score'] > game_scores_23_24['Away_Score']).astype(int)

game_scores_23_24['Visitor'] = game_scores_23_24['Visitor'].map(team_name_dict)

game_scores_23_24['Home'] = game_scores_23_24['Home'].map(team_name_dict)



X_24_25 = game_scores_24_25.copy()

X_24_25 = X_24_25.merge(
    team_data_24_25.add_prefix('home_'),
    left_on='Home',
    right_on='home_team',
    how='inner'
)

X_24_25 = X_24_25.merge(
    team_data_24_25.add_prefix('away_'),
    left_on='Visitor',
    right_on='away_team',
    how='left'
)

y_24_25 = (X_24_25['Home_Score'] > X_24_25['Away_Score']).astype(int)

X_24_25 = X_24_25.drop(columns=dropped_items)



X_23_24 = game_scores_23_24.copy()

X_23_24 = X_23_24.merge(
    team_data_23_24.add_prefix('home_'),
    left_on='Home',
    right_on='home_team',
    how='inner'
)


X_23_24 = X_23_24.merge(
    team_data_23_24.add_prefix('away_'),
    left_on='Visitor',
    right_on='away_team',
    how='left'
)


y_23_24 = (X_23_24['Home_Score'] > X_23_24['Away_Score']).astype(int)

X_23_24 = X_23_24.drop(columns=dropped_items)



X = pd.concat([X_23_24, X_24_25])
y = pd.concat([y_23_24, y_24_25])


# split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#print(X.iloc[0,:])
# Initialize the model clearly
model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=15,
    learning_rate=0.01,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Predict using your model
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")

# Detailed classification report
print(classification_report(y_test, y_pred))


FileNotFoundError: [Errno 2] No such file or directory: '/content/2024_25_NHL_Team_Stats.csv'